# Phase 7 — Đánh giá retrieval và câu trả lời

Notebook canonical của Phase 7: kiểm tra ground truth 104 câu, giải thích metric,
minh họa bằng ranking tính tay (không phải bằng chứng RAG), probe thật trên active
Qdrant (read-only), đọc exact retrieval artifacts đã hoàn thành và recompute summary.

**Không gọi lại paid batch khi Run All.** Answer section chỉ đọc package khớp
dataset/config checksum hiện tại; nếu package chưa tồn tại (gold changed, paid
rerun chưa được authorize) cell dừng rõ ràng, không chọn nhầm package cũ.

Chạy từ repo root:

```bash
uv run --env-file .env python -m jupyter nbconvert --execute --to notebook --inplace notebooks/07_evaluation.ipynb
```


In [ ]:
import json
import nbformat  # noqa: F401  (kept out of runtime import list; sanity only)
import sys
from pathlib import Path

BACKEND = Path("../backend").resolve() if Path("../backend").exists() else Path("backend").resolve()
print("backend:", BACKEND)
sys.path.insert(0, str(BACKEND))


## 1. Ground truth: 104 test cases với gold source/section evidence

Cơ sở: `knowledge-base-hue/foods/evaluation/tests.jsonl` (104 câu, `case_id` `foods-NNNN`).
Mỗi case có `relevant_sources` (tối thiểu nhưng đủ) và `relevant_sections` (source -> section
khai báo); với mỗi cặp (source, section) là một evidence unit, source không khai báo section
thì mỗi mục của source là một unit (mọi section trong source đều có thể match).

Loader strict: nếu dataset sai (thiếu field, section không tồn tại, case_id trùng, count
lệch) **fail ngay**, không skip âm thầm.

In [ ]:
from evaluation.test_loader import load_dataset

REPO = Path(BACKEND).parent.resolve()  # repo root (backend/../)
dataset = load_dataset(
    REPO / "knowledge-base-hue" / "foods" / "evaluation" / "tests.jsonl",
    kb_root=REPO / "knowledge-base-hue",
    expected_count=104,
)
print("cases:", len(dataset.cases))
print("dataset_checksum:", dataset.dataset_checksum)
print("case_id range:", dataset.cases[0].case_id, "->", dataset.cases[-1].case_id)
print("evidence declarations:", sum(len(c.relevant_sources) for c in dataset.cases))

### 1.1 Bảng audit 104 mappings (exact source/section)

Bảng dưới liệt kê **toàn bộ 104 case** với exact evidence unit: mỗi unit là
`(source_path, section)` mà gold yêu cầu retrieval phải tìm thấy. Người dùng
audit bằng cách mở từng source trong `knowledge-base-hue/foods/` và đối chiếu
với `reference_answer` của case (mọi claim trong reference phải được unit đã
khai báo hỗ trợ; unit thừa/trùng/nhóm section không hỗ trợ claim là sai).

In [ ]:
from collections import Counter


def print_table(headers, rows, widths=None):
    rows = [[str(c) for c in r] for r in rows]
    if not widths:
        widths = [max(len(headers[i]), max(len(r[i]) for r in rows)) for i in range(len(headers))]
    def line(values):
        return "  ".join(str(v).ljust(w) for v, w in zip(values, widths))
    print(line(headers))
    print("  ".join("-" * w for w in widths))
    for r in rows:
        print(line(r))


def units_of(c):
    # Exact evidence units of one case as human-readable strings.
    if c.relevant_sections:
        return [f"{src} :: {section}"
                for src, secs in c.relevant_sections.items() for section in secs]
    return [f"{src} :: <any section> (no section declared)" for src in c.relevant_sources]


rows = []
for c in dataset.cases:
    units = units_of(c)
    rows.append((c.case_id, c.category, c.question, str(len(units)), " | ".join(units)))
print_table(
    ("case_id", "category", "question", "units", "exact source :: section"),
    [(r[0], r[1], r[2][:40], r[3], r[4]) for r in rows],
    widths=(12, 15, 42, 6, 78),
)
counts = Counter(r[1] for r in rows)
print("=" * 160)
print("per-category:", " ".join(f"{cat}={counts[cat]}" for cat in sorted(counts)))
assert len(rows) == 104, "expected exactly 104 cases"

## 2. Metrics: Recall@k, MRR@10, nDCG (binary relevance)

- Evidence unit = cặp `(source, section)` khai báo, hoặc `(source, None)` khi source không
  khai báo section.
- Trùng `(source, section)` trong retrieved: chỉ tính rank xuất hiện đầu.
- Relevance là binary 1/0.
- Primary metric: macro-average `Recall@5` của 8 category (mỗi category trọng số bằng
  nhau); secondary là overall trên 104 cases.
- `keyword_coverage@k` chỉ là lexical diagnostic (NFC + casefold + whitespace, exact phrase
  trên title + section + text), **không** thay gold relevance.

### 2.1 Minh họa rank tính tay (không phải bằng chứng RAG)

Kiểm tra metric math bằng ranking giả định đã biết kết quả.

In [ ]:
from core.schema import RetrievedDocument
from evaluation.metrics import case_metrics, keyword_coverage, latency_stats

def doc(source, section, text="x", title="Tên"):
    return RetrievedDocument(id=f"{source}|{section}", score=0.5, text=text,
                             metadata={"source": source, "section": section, "title": title})

A = "foods/restaurants/quan a.md"
B = "foods/restaurants/quan b.md"
hand = [
    doc(A, "Thông tin"),          # rank 1 (relevant)
    doc(A, "Thông tin"),          # duplicate (source, section) -> ignored
    doc(B, "Tóm tắt"),            # rank 3 -> not relevant (gold: only A)
]
m = case_metrics(hand, [A], {A: ["Thông tin"]})
print("hand-calculated: recall@1=1.0 recall@3=1.0 mrr=1.0 ndcg@5=1.0 first_relevant_rank=1")
print("observed:", m["recall_at_1"], m["recall_at_3"], m["mrr_at_10"], round(m["ndcg_at_5"], 6))
assert m["recall_at_1"] == 1.0 and m["mrr_at_10"] == 1.0 and m["first_relevant_rank"] == 1

# second-rank relevant: dcg = 1/log2(3) / idcg 1/log2(2)
hand2 = [doc(B, "Giá"), doc(A, "Thông tin")]
m2 = case_metrics(hand2, [A], {A: ["Thông tin"]})
print("second rank:", m2["recall_at_1"], round(m2["ndcg_at_5"], 6), "expected ndcg=0.63093")
assert m2["recall_at_1"] == 0.0 and abs(m2["ndcg_at_5"] - 0.63093) < 1e-4
print("latency median/p95:", latency_stats([44, 50, 60, 75]))


## 3. Live Qdrant read-only check + một retrieval probe thật

Kiểm tra active collection: chỉ đọc, không reset/reindex/upsert/delete. Probe dùng profile
`dense_only` trên collection `hue_foods_e5_small_384`.

In [ ]:
from core.settings_loader import load_settings
from vectorstore.qdrant import get_client

settings = load_settings()
db = settings["vector_database"]
client = get_client(db["url"], db["timeout"])
assert client.collection_exists(db["collection_name"]), "active collection missing"
count = client.count(db["collection_name"], exact=True).count
echo = client.get_collection(db["collection_name"])
print("collection:", db["collection_name"], "points:", count)
assert count == 572, f"expected 572 points, got {count}"

from core.startup import build_retrieval_stack
from retrieval.service import RetrievalService
import copy
probe_settings = copy.deepcopy(settings)
probe_settings["active_profile"] = "dense_only"
stack = build_retrieval_stack(probe_settings, client=client)
service = RetrievalService(stack)
docs = service.search("bún bò Mệ Kéo ở đâu?")
print("probe returned:", len(docs), "documents")
for d in docs[:3]:
    print(" ", d.id, round(d.score, 4), d.metadata.get("source"), "|", d.metadata.get("section"))
assert docs


## 4. Đọc exact completed retrieval artifacts + recompute summary

Chọn **một exact reviewed package**: 3 retrieval runs (mỗi profile một run) mà:

- `dataset_checksum` == dataset hiện tại (khớp `tests.jsonl` vừa load);
- `summary.status == "complete"` và đủ 104 records unique, mọi record `complete`;
- chung `corpus_checksum`, chung `collection_name`, chung `embedding_model`;
- `config_checksum` của mỗi run == fingerprint profile tương ứng (dùng cùng
  công thức `core.startup._semantic_config` — chống ghép nhầm cấu hình);
- recompute `aggregate_metrics` từ per-case records và **khớp summary file**.

Thiếu bất kỳ thành phần nào hoặc mismatch → **fail rõ ràng**, không pass im lặng.
Nếu chưa có package hoàn chỉnh cho checksum hiện tại, cell in diagnostic đầy đủ
(từng run tồn tại với status/cases/override) và dừng với SystemExit.

In [ ]:
from evaluation import artifacts, metrics as metrics_mod
from evaluation.retrieval_eval import config_fingerprint

RESULTS = BACKEND / "evaluation" / "results"
expected_profiles = ["dense_only", "hybrid_no_rerank", "hybrid_rerank"]

def retrieval_candidates():
    out = {}
    for path in sorted((RESULTS / "retrieval").glob("*.summary.json")):
        out.setdefault(path.read_text(encoding="utf-8"), []).append(path)
    return out

candidates = retrieval_candidates()
by_profile = {}
for _raw, paths in candidates.items():
    for p in paths:
        data = json.loads(p.read_text(encoding="utf-8"))
        if data.get("dataset_checksum") == dataset.dataset_checksum:
            by_profile.setdefault(data["profile"], []).append(data)
completed = {p: [s for s in by_profile.get(p, []) if s.get("status") == "complete"]
             for p in expected_profiles}
missing = [p for p in expected_profiles if len(completed.get(p, [])) != 1]
if missing:
    print(f"[retrieval] NO exact completed package for dataset "
          f"{dataset.dataset_checksum[:8]} (missing/ambiguous: {missing})")
    for data in by_profile.values():
        for s in data:
            print(f"  run={s['run_id']} status={s['status']} "
                  f"cases={s.get('completed_case_count')}/104 "
                  f"override={s.get('evaluation_override')} "
                  f"dataset={s.get('dataset_checksum', '')[:8]}")
    raise SystemExit(
        "exactly one completed retrieval run per profile required; "
        "rerun the 3 full retrieval profiles with the current dataset to "
        "produce the reviewed package (read-only, no paid calls)")
print("[retrieval] exact package found for dataset", dataset.dataset_checksum[:8])

corpus = {completed[p][0]["corpus_checksum"] for p in expected_profiles}
assert len(corpus) == 1, f"profiles must share one corpus checksum, got {corpus}"
common_corpus = corpus.pop()
for p in expected_profiles:
    s = completed[p][0]
    assert s["collection_name"] == settings["vector_database"]["collection_name"], s["run_id"]
    assert s["embedding_model"] == settings["embedding"]["model"], s["run_id"]
    cfg_expected = config_fingerprint(settings, p)
    assert s["config_checksum"] == cfg_expected, (
        f"{p}: run config {s['config_checksum'][:12]} != expected {cfg_expected[:12]}")
print("identity OK: common corpus", common_corpus[:8],
      "| collection/model/profile-config checked")

recomputed = {}
for p in expected_profiles:
    s = completed[p][0]
    run_id = s["run_id"]
    records = artifacts.read_records(RESULTS / "retrieval" / f"{run_id}.jsonl")
    ids = [r["case_id"] for r in records]
    assert len(ids) == 104 and len(set(ids)) == 104, f"{run_id}: expected 104 unique rows"
    assert all(r.get("status") == "complete" for r in records), run_id
    for r in records:
        assert r["dataset_checksum"] == dataset.dataset_checksum, run_id
        assert r["corpus_checksum"] == common_corpus, run_id
        assert r["config_checksum"] == s["config_checksum"], run_id
        assert r["collection_name"] == settings["vector_database"]["collection_name"], run_id
        assert r["embedding_model"] == settings["embedding"]["model"], run_id
    agg = metrics_mod.aggregate_metrics([
        {"case_id": r["case_id"], "category": r["category"], "status": r["status"],
         "metrics": r["metrics"], "latency_ms": r["latency_ms"]} for r in records
    ])
    expected = s["metrics"]
    assert abs(agg["overall"]["recall_at_5"] - expected["overall"]["recall_at_5"]) < 1e-9, run_id
    assert abs(agg["macro_recall_at_5"] - expected["macro_recall_at_5"]) < 1e-9, run_id
    assert agg["cases_total"] == expected["cases_total"] == 104
    recomputed[p] = agg
    print(f"{p}: recall@1={agg['overall']['recall_at_1']:.3f} "
          f"@3={agg['overall']['recall_at_3']:.3f} "
          f"@5={agg['overall']['recall_at_5']:.3f} "
          f"@10={agg['overall']['recall_at_10']:.3f} "
          f"mrr@10={agg['overall']['mrr_at_10']:.3f} "
          f"ndcg@5={agg['overall']['ndcg_at_5']:.3f} "
          f"macro@5={agg['macro_recall_at_5']:.3f}")

### 4.1 So sánh 3 profiles (từ artifact; không tuyên bố winner — Phase 8 mới chọn)

In [ ]:
comp_rows = []
for profile, agg in recomputed.items():
    comp_rows.append((
        profile,
        f"{agg['overall']['recall_at_5']:.4f}",
        f"{agg['macro_recall_at_5']:.4f}",
        f"{agg['overall']['mrr_at_10']:.4f}",
        f"{agg['overall']['ndcg_at_5']:.4f}",
        f"{agg['overall']['ndcg_at_10']:.4f}",
    ))
print_table(("profile", "Recall@5", "macro Recall@5", "MRR@10", "nDCG@5", "nDCG@10"), comp_rows)

## 5. 24 câu trả lời: generation + judge (package vừa chọn)

Chỉ đọc artifacts thuộc đúng package đã kiểm ở bước 4 (dataset/config/model/run
linkage); không gọi lại paid. Cell chọn **one exact answer package**:

- manifest `answer_subset_v1.json` có đúng 24 case_id (mỗi category 3);
- generation run: dataset + `answer_model` + `config_checksum` (profile)
  + `GENERATION_PROMPT_HASH` khớp, **đúng một** run;
- judge run liên kết `judge-<suffix>`, mọi judge row khớp
  model/rubric/prompt/dataset/config checksum, mọi `generation_run_id` thuộc
  generation run này;
- calibration package được validate bằng `validate_calibration_package`
  (8 rows + summary gate + checksum + prompt hash);
- summary `summaries/<judge_run>.json` liên kết đúng 2 run và recompute counts
  khớp; bảng hiển thị **cả các row failed/error**, không drop khỏi denominator.

In [ ]:
# Cell 15 — Answer Generation and LLM-as-Judge Evidence
# Loads frozen generation/judge runs for hybrid_rerank (the Phase 7 gate profile),
# validates exact calibration package linkage, raw row identity, and summary counts.

from pathlib import Path
import json
import hashlib
from core.settings_loader import load_settings
from evaluation import artifacts, answer_eval
from evaluation.test_loader import load_dataset
from evaluation.retrieval_eval import config_fingerprint, compute_corpus_checksum
from vectorstore.qdrant import get_client

REPO = globals().get("REPO") or Path.cwd().resolve()
if REPO.name in ("notebooks", "backend"):
    REPO = REPO.parent
RESULTS = globals().get("RESULTS") or (REPO / "backend" / "evaluation" / "results")
settings = globals().get("settings") or load_settings()
dataset = globals().get("dataset") or load_dataset((REPO / "backend" / settings["evaluation"]["test_file"]).resolve(),
                                                    kb_root=(REPO / "knowledge-base-hue").resolve(),
                                                    expected_count=104)

# Validate 24-case subset manifest
manifest_path = REPO / "knowledge-base-hue" / "foods" / "evaluation" / "answer_subset_v1.json"
assert manifest_path.exists(), f"answer subset manifest missing: {manifest_path}"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_ids = [c["case_id"] if isinstance(c, dict) else c for c in manifest["cases"]]
assert len(manifest_ids) == 24, f"expected 24 manifest cases, got {len(manifest_ids)}"
manifest_cases = {c.case_id: c for c in dataset.cases if c.case_id in set(manifest_ids)}
assert len(manifest_cases) == 24, "manifest cases missing from dataset"

# Identify latest generation run for hybrid_rerank bound to current dataset
gen_dir = RESULTS / "generations"
expected_gen_cfg = config_fingerprint(settings, "hybrid_rerank")
client = globals().get("client") or get_client(settings["vector_database"]["url"], settings["vector_database"]["timeout"])
common_corpus = globals().get("common_corpus") or compute_corpus_checksum(settings, client)

gen_run_ids = []
if gen_dir.exists():
    for path in sorted(gen_dir.glob("generation-*-hybrid_rerank-*.jsonl"), reverse=True):
        if path.name.endswith(".partial.jsonl"):
            continue
        rows = artifacts.read_records(path)
        if rows and rows[0].get("dataset_checksum") == dataset.dataset_checksum:
            gen_run_ids.append(path.stem)

if not gen_run_ids:
    # List available runs for diagnosis
    available = sorted(gen_dir.glob("generation-*-hybrid_rerank-*.jsonl")) if gen_dir.exists() else []
    print("[diagnostic] Available generation runs on disk:")
    for path in available:
        rows = artifacts.read_records(path)
        if rows:
            r0 = rows[0]
            print(f"  generation run={path.stem} dataset={str(r0.get('dataset_checksum'))[:8]} "
                  f"model={r0.get('answer_model')} cfg={str(r0.get('config_checksum'))[:8]}")
    raise SystemExit(
        "answer generation/judge package for the current dataset is missing: the gold "
        "audit changed tests.jsonl, so frozen generation/judge runs are bound to the "
        "previous dataset version; a paid calibration + 24-case rerun on the new checksum "
        "requires user authorization (Phase 7 guide handoff). Retrieval evidence above "
        "is current; answer metrics stay diagnostic until the authorized rerun.")
gen_run = gen_run_ids[0]
judge_run = "judge-" + gen_run[len("generation-"):]

# The package's summary links the exact calibration package (not just "latest").
answer_summary = json.loads(
    (RESULTS / "summaries" / f"{judge_run}.json").read_text(encoding="utf-8"))
assert answer_summary["generation_run_id"] == gen_run
assert answer_summary["judge_run_id"] == judge_run
assert answer_summary["dataset_checksum"] == dataset.dataset_checksum
assert answer_summary["config_checksum"] == expected_gen_cfg
assert answer_summary["collection_name"] == settings["vector_database"]["collection_name"]
assert answer_summary["cases_total"] == 24

# Mandatory budget artifact verification
budget_path_rel = answer_summary.get("budget_artifact_path")
assert budget_path_rel and isinstance(budget_path_rel, str), "budget_artifact_path missing or invalid in summary"
assert not budget_path_rel.startswith("/") and not ".." in budget_path_rel, f"invalid budget path: {budget_path_rel}"
budget_file = RESULTS / budget_path_rel
assert budget_file.exists(), f"budget file missing: {budget_file}"
assert hashlib.sha256(budget_file.read_bytes()).hexdigest() == answer_summary.get("budget_artifact_checksum"), "budget checksum mismatch"

# Load budget state and validate full identity and totals
budget_obj = answer_eval.CallBudget.load(
    budget_file,
    expected_identity={
        "dataset_checksum": dataset.dataset_checksum,
        "config_checksum": expected_gen_cfg,
        "corpus_checksum": common_corpus,
        "collection_name": settings["vector_database"]["collection_name"],
        "answer_profile": "hybrid_rerank",
        "answer_model": settings["llm"]["answer_model"],
        "judge_model": settings["evaluation"]["judge_model"],
        "generation_prompt_hash": answer_eval.GENERATION_PROMPT_HASH,
        "rubric_version": answer_eval.RUBRIC_VERSION,
        "rubric_prompt_hash": answer_eval.RUBRIC_PROMPT_HASH,
    },
)
assert budget_obj.package_id == gen_run
assert answer_summary["provider_calls_total"] == budget_obj.calls
assert abs(float(answer_summary["provider_cost_usd_total"]) - float(budget_obj.effective_cost_usd)) < 1e-8
assert answer_summary["unresolved_attempt_count"] == sum(1 for a in budget_obj.attempts if a.get("status") == "reserved")
assert 56 <= budget_obj.calls <= 64, f"expected between 56 and 64 calls, got {budget_obj.calls}"

cal_run = answer_summary["calibration_run_id"]
cal_summary_path = RESULTS / "summaries" / f"{cal_run}.json"
cal_rows_path = RESULTS / "judges" / f"{cal_run}.jsonl"
assert cal_summary_path.exists(), f"calibration summary missing: {cal_summary_path}"
assert cal_rows_path.exists(), f"calibration rows missing: {cal_rows_path}"
cal_data = json.loads(cal_summary_path.read_text(encoding="utf-8"))
cal_rows = artifacts.read_records(cal_rows_path)
assert len(cal_rows) == 8, f"expected 8 calibration rows, got {len(cal_rows)}"
assert len({r.get("generation_run_id") for r in cal_rows}) == 8
samples_path = REPO / "knowledge-base-hue" / "foods" / "evaluation" / "judge_calibration_v1.jsonl"
samples = answer_eval.load_calibration_samples(str(samples_path))
answer_eval.validate_calibration_package(
    cal_rows, samples,
    dataset_checksum=dataset.dataset_checksum,
    config_checksum=expected_gen_cfg,
    samples_checksum=answer_eval.calibration_samples_checksum(str(samples_path)),
    judge_model=settings["evaluation"]["judge_model"],
    rubric_version=answer_eval.RUBRIC_VERSION,
    prompt_hash=answer_eval.RUBRIC_PROMPT_HASH,
    summary=cal_data)
print(f"[answers] package {gen_run} + {judge_run}; calibration {cal_run} "
      f"validated (gate passed)")

raw_gen_records = artifacts.read_records(RESULTS / "generations" / f"{gen_run}.jsonl")
assert len(raw_gen_records) == 24, f"expected 24 generation rows, got {len(raw_gen_records)}"
assert len({r.get("case_id") for r in raw_gen_records}) == 24, "duplicate generation case_id detected"
assert set(r.get("case_id") for r in raw_gen_records) == set(manifest_ids), "generation rows must match manifest ids"
for r in raw_gen_records:
    case_id = r.get("case_id")
    case = manifest_cases[case_id]
    assert r.get("run_id") == gen_run, f"unexpected generation run_id in row: {r.get('run_id')}"
    assert r.get("dataset_checksum") == dataset.dataset_checksum
    assert r.get("corpus_checksum") == common_corpus
    assert r.get("config_checksum") == expected_gen_cfg
    assert r.get("collection_name") == settings["vector_database"]["collection_name"]
    assert r.get("calibration_run_id") == cal_run
    assert r.get("answer_model") == settings["llm"]["answer_model"]
    assert r.get("prompt_hash") == answer_eval.GENERATION_PROMPT_HASH
    assert r.get("category") == case.category, f"{case_id}: category mismatch"
    assert r.get("question") == case.question, f"{case_id}: question mismatch"
    assert r.get("reference_answer") == case.reference_answer, f"{case_id}: reference mismatch"

gen_records = {r["case_id"]: r for r in raw_gen_records}

raw_judge_records = artifacts.read_records(RESULTS / "judges" / f"{judge_run}.jsonl")
expected_gids = {f"{gen_run}:{cid}" for cid in manifest_ids}
assert len(raw_judge_records) == 24, f"expected 24 judge rows, got {len(raw_judge_records)}"
assert len({r.get("generation_run_id") for r in raw_judge_records}) == 24, "duplicate judge generation_run_id detected"
assert set(r.get("generation_run_id") for r in raw_judge_records) == expected_gids, "judge rows must match expected generation ids"
for r in raw_judge_records:
    gid = r.get("generation_run_id")
    case_id = r.get("case_id")
    assert gid == f"{gen_run}:{case_id}", f"judge row gid {gid} does not match case_id {case_id}"
    case = manifest_cases[case_id]
    assert r.get("run_id") == judge_run, f"unexpected judge run_id: {r.get('run_id')}"
    assert r.get("category") == case.category, f"{case_id}: judge category mismatch"
    assert r.get("judge_model") == settings["evaluation"]["judge_model"]
    assert r.get("rubric_version") == answer_eval.RUBRIC_VERSION
    assert r.get("prompt_hash") == answer_eval.RUBRIC_PROMPT_HASH
    assert r.get("dataset_checksum") == dataset.dataset_checksum
    assert r.get("corpus_checksum") == common_corpus
    assert r.get("config_checksum") == expected_gen_cfg
    assert r.get("collection_name") == settings["vector_database"]["collection_name"]
    assert r.get("calibration_run_id") == cal_run

judge_records = {r["generation_run_id"]: r for r in raw_judge_records}

gen_complete = {c: r for c, r in gen_records.items() if r.get("status") == "complete"}
judge_complete = {g: r for g, r in judge_records.items() if r.get("status") == "complete"}
passed = sum(1 for r in judge_complete.values() if answer_eval.judge_passes(r["scores"]))
assert answer_summary["completed_generation"] == len(gen_complete)
assert answer_summary["failed_generation"] == len(gen_records) - len(gen_complete)
assert answer_summary["completed_judge"] == len(judge_complete)
assert answer_summary["failed_judge"] == len(gen_records) - len(judge_complete)
assert answer_summary["passed"] == passed
assert answer_summary["status"] == "complete"
assert answer_summary["cases_total"] == 24

result_rows = []
for case_id in sorted(gen_records):
    if gen_records[case_id].get("status") != "complete":
        result_rows.append((case_id, gen_records[case_id]["question"][:30],
                            "gen " + (gen_records[case_id].get("status") or "?"),
                            gen_records[case_id].get("error_type") or "-",
                            "-", "-", "-", "generation row kept in artifact"))
        continue
    jrow = judge_records.get(f"{gen_run}:{case_id}")
    if jrow is None:
        result_rows.append((case_id, gen_records[case_id]["question"][:30],
                            "judge missing", "-", "-", "-", "-",
                            "judge row missing - kept visible"))
        continue
    if jrow.get("status") != "complete":
        result_rows.append((case_id, gen_records[case_id]["question"][:30],
                            "judge " + (jrow.get("status") or "?"),
                            jrow.get("error_type") or "-", str(jrow.get("attempts", 1)),
                            "-", "-", "failed judge row kept in artifact"))
        continue
    g = gen_records[case_id]
    result_rows.append((
        case_id, g["question"][:30],
        g["generated_answer"][:38],
        ", ".join(g["used_sources"])[:34],
        str(jrow["scores"]["accuracy"]), str(jrow["scores"]["groundedness"]),
        "PASS" if answer_eval.judge_passes(jrow["scores"]) else "fail",
        jrow["feedback"][:40],
    ))
print_table(
    ("case_id", "question", "answer/status", "sources", "acc", "grn", "pass", "feedback"),
    result_rows,
    widths=(12, 32, 40, 36, 5, 5, 6, 42),
)
print("rows shown:", len(result_rows),
      "| status:", answer_summary["status"],
      "| generation:", answer_summary["completed_generation"],
      "| judged:", answer_summary["completed_judge"],
      "| passed:", answer_summary["passed"],
      "| provider calls:", answer_summary.get("provider_calls_total", "-"),
      "| provider cost: $", f"{answer_summary.get('provider_cost_usd_total', 0.0):.4f}")


## 6. Ghi chú

- Artifacts: `backend/evaluation/results/{retrieval,generations,judges,summaries}/`.
- `retrieval/` records có `dataset_checksum`, `corpus_checksum`, `config_checksum`,
  run_id, latency, metrics; run subset (`--max-cases`), partial hoặc gắn checksum
  dataset cũ chỉ là diagnostic, không dùng làm comparison evidence.
- Không có fake/replay fallback; mọi số liệu trên là observed từ run thật.
- Không gọi paid API khi Run All. **Answer package hiện tại
  (`generation-…-5c6ba589`) gắn dataset cũ — bị superseded vì gold audit đổi
  `tests.jsonl`; paid calibration + 24-case rerun trên checksum hiện tại cần
  user authorization trước khi chạy.**
- Key không lưu trong notebook; env nạp qua `uv run --env-file .env`.
